# Hospital Readmission Reduction Program (HRRP) Analysis

This project analyzes Hospital Readmissions Reduction Program (HRRP) data to explore patterns in hospital performance and identify factors associated with higher readmission rates.

The analysis focuses on data cleaning, validation, and exploratory analysis to better understand variation across hospitals and conditions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


All good!


## 1. Load HRRP Dataset 

The dataset is loaded from the raw directory. This file contains hospital-level information related to readmission performance and associated metrics.

The raw dataset is preserved in its original form to maintain data integrity.

## 2. Initial Data Inspection

Before performing any transformations, the dataset is explored to understand its structure, including the number of rows and columns, data types, and the presence of missing values.

In [2]:
df = pd.read_csv("../data/raw/FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv")

df.head()

,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137.0,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273.0,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122.0,NaN,0.9823,16.6384,16.9389,19,07/01/2021,06/30/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507.0,NaN,0.9871,15.7529,15.9591,79,07/01/2021,06/30/2024


In [7]:
df.shape

(18330, 12)

In [4]:
df.columns

Index(['Facility Name', 'Facility ID', 'State', 'Measure Name',
       'Number of Discharges', 'Footnote', 'Excess Readmission Ratio',
       'Predicted Readmission Rate', 'Expected Readmission Rate',
       'Number of Readmissions', 'Start Date', 'End Date'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18330 entries, 0 to 18329
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Facility Name               18330 non-null  str    
 1   Facility ID                 18330 non-null  int64  
 2   State                       18330 non-null  str    
 3   Measure Name                18330 non-null  str    
 4   Number of Discharges        8242 non-null   float64
 5   Footnote                    6987 non-null   float64
 6   Excess Readmission Ratio    11720 non-null  float64
 7   Predicted Readmission Rate  11720 non-null  float64
 8   Expected Readmission Rate   11720 non-null  float64
 9   Number of Readmissions      11720 non-null  str    
 10  Start Date                  18330 non-null  str    
 11  End Date                    18330 non-null  str    
dtypes: float64(5), int64(1), str(6)
memory usage: 1.7 MB


In [6]:
df.isnull().sum()

Facility Name                     0
Facility ID                       0
State                             0
Measure Name                      0
Number of Discharges          10088
Footnote                      11343
Excess Readmission Ratio       6610
Predicted Readmission Rate     6610
Expected Readmission Rate      6610
Number of Readmissions         6610
Start Date                        0
End Date                          0
dtype: int64

## 3. Initial Data Observations

This dataset contains 18,330 records across 12 columns, representing hospital-level readmission metrics across multiple conditions.

Several data quality issues were identified:

- Significant missing values in key analytical fields such as discharge counts and readmission metrics
- Non-numeric values (e.g., "Too Few to Report") in the number of readmissions column
- Multiple rows per facility corresponding to different clinical measures

These issues will need to be addressed before proceeding with analysis to ensure accurate and reliable results.

In [8]:
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w\s]", "", regex=True)
)

df.columns

Index(['facility_name', 'facility_id', 'state', 'measure_name',
       'number_of_discharges', 'footnote', 'excess_readmission_ratio',
       'predicted_readmission_rate', 'expected_readmission_rate',
       'number_of_readmissions', 'start_date', 'end_date'],
      dtype='str')

## 4. Convert Data Types

Several columns require data type adjustments before analysis.

- `number_of_readmissions` is converted to numeric so it can be used in calculations
- `start_date` and `end_date` are converted to datetime format to support date-based analysis
- Non-numeric values such as "Too Few to Report" are converted to missing values

In [10]:
df['number_of_readmissions'] = pd.to_numeric(
    df['number_of_readmissions'],
    errors='coerce'
)

df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18330 entries, 0 to 18329
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   facility_name               18330 non-null  str           
 1   facility_id                 18330 non-null  int64         
 2   state                       18330 non-null  str           
 3   measure_name                18330 non-null  str           
 4   number_of_discharges        8242 non-null   float64       
 5   footnote                    6987 non-null   float64       
 6   excess_readmission_ratio    11720 non-null  float64       
 7   predicted_readmission_rate  11720 non-null  float64       
 8   expected_readmission_rate   11720 non-null  float64       
 9   number_of_readmissions      8037 non-null   float64       
 10  start_date                  18330 non-null  datetime64[us]
 11  end_date                    18330 non-null  datetime64[us]
dtypes

## 5. Assess Record Uniqueness

The dataset contains multiple rows per facility bacause each hospital is evaluated across several readmission measures.

To confirm that the data structure is valid, duplicate checks are performed at the facility-measure level.

In [11]:
df[['facility_id', 'measure_name']].duplicated().sum()

np.int64(0)

## 6. Create an Analysis-Ready Dataset

Rows missing the key readmission metrics are removed or analytical purposes.

This creates a cleaner subset of the data that can be used for visualization and comparison across hospitals and measures.

In [12]:
analysis_df = df.dropna(
    subset=[
        'excess_readmission_ratio',
        'predicted_readmission_rate',
        'expected_readmission_rate'
    ]
)

analysis_df.shape

(11720, 12)

## 7. Final Analysis Dataset

After removing rows with missing readmission metrics, the dataset contains 11,720 records.

These rows include complete values for excess readmission ratio, predicted readmission rate, making them suitable for analysis and visualization.

The excluded rows likely represent hospitals or measures with insufficient case volume for reliable reporting.

## 8. Save Cleaned Dataset

The cleaned analysis-ready dataset os saved to the processed data directory.

This allows future notebooks to begin directly fromthe cleaned data without repeating the preparation steps.

In [13]:
analysis_df.to_csv(
    "../data/processed/hrrp_readmissions_cleaned.csv",
    index=False
)